In [13]:
"""QA test/train 통계: 논문 표에 넣을 값(셀)용 집계만 print."""

from __future__ import annotations

from pathlib import Path
import csv
import json
from collections import defaultdict, Counter

In [14]:
def resolve_qa_dir() -> Path:
    """check_train_test.ipynb 실행 위치와 무관하게 ace_safe_ver/QA 를 찾음."""
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd / "ace_safe_ver" / "QA",
        cwd.parent / "QA" if cwd.name == "ace_safe_ver" else None,
    ]
    for c in candidates:
        if c and c.is_dir() and (c / "test").is_dir() and (c / "train").is_dir():
            return c.resolve()
    for anc in [cwd, *cwd.parents]:
        q = anc / "ace_safe_ver" / "QA"
        if q.is_dir() and (q / "test").is_dir() and (q / "train").is_dir():
            return q.resolve()
    return cwd.resolve()


QA_DIR = resolve_qa_dir()
SPLITS = ["test", "train"]


def resolve_splits_dir() -> Path:
    """merged_train.csv / merged_test.csv 위치 (벤치마크 split 정의)."""
    cwd = Path.cwd()
    for anc in [cwd, *cwd.parents]:
        p = anc / "ace_safe_ver" / "splits" / "scaffold_by_endpoint_unseen_ver"
        if (p / "merged_train.csv").is_file() and (p / "merged_test.csv").is_file():
            return p.resolve()
    return (cwd / "ace_safe_ver" / "splits" / "scaffold_by_endpoint_unseen_ver").resolve()


SPLITS_DIR = resolve_splits_dir()


def _load_merged_csv(name: str) -> list[dict]:
    p = SPLITS_DIR / name
    if not p.is_file():
        return []
    with open(p, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


_train_csv = _load_merged_csv("merged_train.csv")
_test_csv = _load_merged_csv("merged_test.csv")

print("=" * 80)
print("(0) 벤치마크 split — merged_train.csv / merged_test.csv (분자 쌍 = 행 1줄)")
print("-" * 80)
print(f"SPLITS_DIR = {SPLITS_DIR}")
if not _train_csv and not _test_csv:
    print("  (CSV 없음 — 경로 확인)")
else:
    print(f"Train rows: {len(_train_csv):,}")
    print(f"Test rows:  {len(_test_csv):,}")
    print(f"Total:      {len(_train_csv) + len(_test_csv):,}")
    for label, rows in [("Train", _train_csv), ("Test", _test_csv)]:
        by_ds = Counter((r.get("dataset_name") or "").strip() or "(empty)" for r in rows)
        ep_by_ds: dict[str, set] = defaultdict(set)
        for r in rows:
            ds = (r.get("dataset_name") or "").strip() or "(empty)"
            ep = r.get("endpoint")
            if ep:
                ep_by_ds[ds].add(str(ep))
        print(f"\n--- {label}: dataset_name별 행 수 / 고유 endpoint 수 ---")
        for ds in sorted(by_ds.keys(), key=lambda x: (-by_ds[x], x)):
            print(f"  {ds:20s}  rows={by_ds[ds]:6,}  endpoints={len(ep_by_ds[ds]):3d}")

# QA 스캔 범위: both_repre만 (단일 representation 벤치마크), agentic_flow_qa 제외
EXCLUDE_AGENTIC = True

print("\n" + "=" * 80)
print("(이하 QA) 경로 필터: both_repre 포함만 | agentic_flow_qa " + ("제외" if EXCLUDE_AGENTIC else "포함"))
print("QA_DIR =", QA_DIR)
print("=" * 80)

# ===== 논문 표에 맞춘 '헤더 이름' 정의 (필요하면 여기만 바꿔도 됨) =====
ROW_HEADERS = {
    "dataset": "Dataset (dataset_name)",
    "endpoint": "Endpoint",
    "molecule": "Molecule pair (source_index)",
    "qa": "# QA (jsonl rows)",
}

COLUMN_HEADERS = {
    "split": "Split",
    "task": "Task",
    "repr": "Molecule representation",
    "step": "Reasoning setting",
}

TASK_GROUPS = {
    "task1": "Task 1: Toxic fragment identification",
    "task2": "Task 2: Non-toxic fragment generation",
    "task3": "Task 3: Non-toxic SMILES generation",
    "task3_instruction": "Task 3 (instruction-based)",
    "task3_stepwise_cot": "Task 3 (stepwise CoT)",
    "subtask1": "Subtask 1",
    "subtask2": "Subtask 2",
    "other": "Other",
}


def infer_task_group(parts: tuple[str, ...]) -> str:
    # parts 예: (split, task_root, repr, step, file)
    # agentic_flow_qa 아래는 실제 task가 다음 세그먼트
    task_root = parts[1] if len(parts) > 1 else ""
    task = parts[2] if task_root == "agentic_flow_qa" and len(parts) > 2 else task_root

    if task.startswith("task1_"):
        return "task1"
    if task.startswith("task2_"):
        return "task2"
    if task.startswith("task3_stepwise_cot_"):
        return "task3_stepwise_cot"
    if task.startswith("task3_instruction_") or task.startswith("task3_CoT_"):
        return "task3_instruction"
    if task.startswith("task3_"):
        return "task3"
    if task.startswith("subtask1_"):
        return "subtask1"
    if task.startswith("subtask2_"):
        return "subtask2"
    return "other"


def infer_repr_step(parts: tuple[str, ...]) -> tuple[str, str]:
    repr_ = "(none)"
    step = "(none)"
    for p in parts:
        if p in ("only_smiles", "only_safe", "both_repre"):
            repr_ = p
        if p in ("single_step", "multi_step"):
            step = p
    return repr_, step


def iter_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue


def add_set(s: set, v):
    if v is None:
        return
    try:
        s.add(v)
    except TypeError:
        pass


def print_block(title: str, lines: list[str]):
    print("\n" + "=" * 80)
    print(title)
    print("-" * 80)
    for ln in lines:
        print(ln)


# ===== 1) both_repre QA jsonl만 스캔 =====
all_files: list[Path] = []
for sp in SPLITS:
    root = QA_DIR / sp
    if not root.is_dir():
        continue
    for fp in sorted(root.rglob("*_qa.jsonl")):
        rel = fp.relative_to(QA_DIR).parts
        if "both_repre" not in rel:
            continue
        if EXCLUDE_AGENTIC and "agentic_flow_qa" in rel:
            continue
        all_files.append(fp)

if not all_files:
    print("No both_repre *_qa.jsonl found under", QA_DIR)
    raise SystemExit(0)

# 집계 컨테이너
by_split = defaultdict(lambda: {"qa": 0, "datasets": set(), "endpoints": set(), "src": set()})
by_split_task = defaultdict(lambda: {"qa": 0, "datasets": set(), "endpoints": set(), "src": set()})
by_split_task_repr_step = defaultdict(int)  # QA rows
by_split_dataset = defaultdict(lambda: {"qa": 0, "endpoints": set(), "src": set()})

# 스캔
for fp in all_files:
    rel_parts = fp.relative_to(QA_DIR).parts
    split = rel_parts[0]
    task_group = infer_task_group(rel_parts)
    repr_, step = infer_repr_step(rel_parts)

    for o in iter_jsonl(fp):
        by_split[split]["qa"] += 1
        by_split_task[(split, task_group)]["qa"] += 1
        if repr_ != "(none)" or step != "(none)":
            by_split_task_repr_step[(split, task_group, repr_, step)] += 1

        ds = o.get("dataset_name")
        ep = o.get("endpoint")
        si = o.get("source_index")

        add_set(by_split[split]["datasets"], ds)
        add_set(by_split[split]["endpoints"], ep)
        add_set(by_split[split]["src"], si)

        add_set(by_split_task[(split, task_group)]["datasets"], ds)
        add_set(by_split_task[(split, task_group)]["endpoints"], ep)
        add_set(by_split_task[(split, task_group)]["src"], si)

        # dataset 중심 집계(논문 표의 'Dataset' 컬럼 대체)
        if ds is not None:
            by_split_dataset[(split, ds)]["qa"] += 1
            add_set(by_split_dataset[(split, ds)]["endpoints"], ep)
            add_set(by_split_dataset[(split, ds)]["src"], si)

# ===== 2) 논문 표처럼 '셀 값'으로 넣을 통계 print =====

# (A) Split 전체 요약
lines = []
for sp in SPLITS:
    if sp not in by_split:
        continue
    s = by_split[sp]
    lines.append(
        f"[{COLUMN_HEADERS['split']}={sp}]  "
        f"{ROW_HEADERS['qa']}={s['qa']:,} | "
        f"{ROW_HEADERS['dataset']}={len(s['datasets']):,} | "
        f"{ROW_HEADERS['endpoint']}={len(s['endpoints']):,} | "
        f"{ROW_HEADERS['molecule']}={len(s['src']):,}"
    )
print_block("(A) Split summary", lines)

# (B) Split × Task 그룹 요약 (논문 표의 'Type / tasks' 영역 대체)
lines = []
for sp in SPLITS:
    for tg_key, tg_name in TASK_GROUPS.items():
        key = (sp, tg_key)
        if key not in by_split_task:
            continue
        s = by_split_task[key]
        if s["qa"] == 0:
            continue
        lines.append(
            f"[{COLUMN_HEADERS['split']}={sp} | {COLUMN_HEADERS['task']}={tg_name}]  "
            f"{ROW_HEADERS['qa']}={s['qa']:,} | "
            f"{ROW_HEADERS['dataset']}={len(s['datasets']):,} | "
            f"{ROW_HEADERS['endpoint']}={len(s['endpoints']):,} | "
            f"{ROW_HEADERS['molecule']}={len(s['src']):,}"
        )
print_block("(B) Split × Task-group summary", lines)

# (C) Split × Task × Repr × Step (논문 표의 'Dimension' 느낌)
# 값은 QA rows만 우선 출력 (원하면 endpoint/src unique도 추가 가능)
lines = []
keys = sorted(by_split_task_repr_step.keys(), key=lambda x: (x[0], x[1], x[2], x[3]))
for sp, tg_key, repr_, step in keys:
    n = by_split_task_repr_step[(sp, tg_key, repr_, step)]
    if n == 0:
        continue
    lines.append(
        f"[{COLUMN_HEADERS['split']}={sp} | {COLUMN_HEADERS['task']}={TASK_GROUPS.get(tg_key, tg_key)} | "
        f"{COLUMN_HEADERS['repr']}={repr_} | {COLUMN_HEADERS['step']}={step}]  "
        f"{ROW_HEADERS['qa']}={n:,}"
    )
print_block("(C) Split × Task × Representation × Step (QA count)", lines)

# (D) Split × Dataset 요약 (논문 표의 'Dataset Name' 행/열 대체)
# dataset_name이 많은 경우가 있어 상위 N개만 보고 싶으면 N만 줄이면 됨.
TOP_N = None  # 예: 20

lines = []
items = []
for (sp, ds), s in by_split_dataset.items():
    items.append((sp, ds, s["qa"], len(s["endpoints"]), len(s["src"])) )
items.sort(key=lambda x: (x[0], -x[2], x[1]))
if TOP_N is not None:
    items = items[:TOP_N]

for sp, ds, qa_n, ep_n, src_n in items:
    lines.append(
        f"[{COLUMN_HEADERS['split']}={sp} | {ROW_HEADERS['dataset']}={ds}]  "
        f"{ROW_HEADERS['qa']}={qa_n:,} | {ROW_HEADERS['endpoint']}={ep_n:,} | {ROW_HEADERS['molecule']}={src_n:,}"
    )
print_block("(D) Split × Dataset_name summary", lines)

# (E) 보너스: 파일 수/커버리지 확인
lines = [
    f"both_repre QA jsonl files: {len(all_files):,}",
    "(only_smiles / only_safe 경로 제외, agentic_flow_qa 제외 시 중복 없음)",
]
print_block("(E) File-level sanity checks", lines)

(0) 벤치마크 split — merged_train.csv / merged_test.csv (분자 쌍 = 행 1줄)
--------------------------------------------------------------------------------
SPLITS_DIR = /Users/jang-wonjun/Desktop/DMISLab/ToxAgent/ace_safe_ver/splits/scaffold_by_endpoint_unseen_ver
Train rows: 57,008
Test rows:  6,243
Total:      63,251

--- Train: dataset_name별 행 수 / 고유 endpoint 수 ---
  tox21_df              rows=34,527  endpoints= 12
  metabolism            rows= 9,030  endpoints=  5
  ames                  rows= 6,780  endpoints=  1
  herg_unified          rows= 4,968  endpoints=  1
  sider                 rows= 1,525  endpoints= 17
  skin_reaction         rows=   107  endpoints=  1
  dilist                rows=    71  endpoints=  1

--- Test: dataset_name별 행 수 / 고유 endpoint 수 ---
  tox21_df              rows= 2,815  endpoints= 12
  sider                 rows= 1,003  endpoints= 27
  metabolism            rows= 1,002  endpoints=  5
  ames                  rows=   710  endpoints=  1
  herg_unified          rows

In [15]:
# --- Dataset Name별 고유 endpoint 개수 (단독 실행 가능) ---
from pathlib import Path
import csv
import json
from collections import defaultdict


def _resolve_qa_dir() -> Path:
    cwd = Path.cwd()
    for c in (cwd, cwd / "ace_safe_ver" / "QA", cwd.parent / "QA" if cwd.name == "ace_safe_ver" else None):
        if c and c.is_dir() and (c / "test").is_dir() and (c / "train").is_dir():
            return c.resolve()
    for anc in [cwd, *cwd.parents]:
        q = anc / "ace_safe_ver" / "QA"
        if q.is_dir() and (q / "test").is_dir() and (q / "train").is_dir():
            return q.resolve()
    return cwd.resolve()


def _iter_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue


# 표 헤더 순서 ↔ JSONL 필드 dataset_name
DATASET_DISPLAY_ORDER = [
    ("hERG", "herg_unified"),
    ("Tox21", "tox21_df"),
    ("Sider", "sider"),
    ("AMES", "ames"),
    ("Clintox", "clintox"),
    ("DICTrank", "dictrank"),
    ("DIRIL", "diril"),
    ("DILIst", "dilist"),
    ("Skin Reaction", "skin_reaction"),
    ("Metabolism", "metabolism"),
]

_qa_dir = _resolve_qa_dir()
_files_by_split: dict[str, list[Path]] = {"train": [], "test": []}
for sp in ("test", "train"):
    r = _qa_dir / sp
    if not r.is_dir():
        continue
    for fp in r.rglob("*_qa.jsonl"):
        rel = fp.relative_to(_qa_dir).parts
        if "both_repre" not in rel:
            continue
        if "agentic_flow_qa" in rel:
            continue
        _files_by_split[sp].append(fp)

# dataset_name → (고유 endpoint), split별 QA 행 수·고유 endpoint
_endpoints_all: dict[str, set] = defaultdict(set)
_endpoints_train: dict[str, set] = defaultdict(set)
_endpoints_test: dict[str, set] = defaultdict(set)
_qa_train: dict[str, int] = defaultdict(int)
_qa_test: dict[str, int] = defaultdict(int)

for sp, fps in _files_by_split.items():
    for fp in fps:
        for o in _iter_jsonl(fp):
            ds = o.get("dataset_name")
            ep = o.get("endpoint")
            if ds is None:
                continue
            ds = str(ds)
            if ep is not None:
                _endpoints_all[ds].add(str(ep))
                if sp == "train":
                    _endpoints_train[ds].add(str(ep))
                else:
                    _endpoints_test[ds].add(str(ep))
            if sp == "train":
                _qa_train[ds] += 1
            else:
                _qa_test[ds] += 1


def _row(label: str, vals: list[int], total: bool = True) -> str:
    s = label + "\t" + "\t".join(str(v) for v in vals)
    if total:
        s += "\t" + str(sum(vals))
    return s


def _resolve_splits_dir_f() -> Path:
    cwd = Path.cwd()
    for anc in [cwd, *cwd.parents]:
        p = anc / "ace_safe_ver" / "splits" / "scaffold_by_endpoint_unseen_ver"
        if (p / "merged_train.csv").is_file() and (p / "merged_test.csv").is_file():
            return p.resolve()
    return (cwd / "ace_safe_ver" / "splits" / "scaffold_by_endpoint_unseen_ver").resolve()


# ----- merged CSV (벤치마크 split = 분자 쌍 1행) -----
_spl = _resolve_splits_dir_f()
_csv_tr_rows: list[dict] = []
_csv_te_rows: list[dict] = []
if (_spl / "merged_train.csv").is_file():
    with open(_spl / "merged_train.csv", newline="", encoding="utf-8") as f:
        _csv_tr_rows = list(csv.DictReader(f))
if (_spl / "merged_test.csv").is_file():
    with open(_spl / "merged_test.csv", newline="", encoding="utf-8") as f:
        _csv_te_rows = list(csv.DictReader(f))

_csv_n_tr = defaultdict(int)
_csv_n_te = defaultdict(int)
_csv_ep_tr = defaultdict(set)
_csv_ep_te = defaultdict(set)
for r in _csv_tr_rows:
    ds = (r.get("dataset_name") or "").strip()
    if not ds:
        continue
    _csv_n_tr[ds] += 1
    ep = r.get("endpoint")
    if ep:
        _csv_ep_tr[ds].add(str(ep))
for r in _csv_te_rows:
    ds = (r.get("dataset_name") or "").strip()
    if not ds:
        continue
    _csv_n_te[ds] += 1
    ep = r.get("endpoint")
    if ep:
        _csv_ep_te[ds].add(str(ep))

internals = [internal for _, internal in DATASET_DISPLAY_ORDER]
_headers = "\t".join(d for d, _ in DATASET_DISPLAY_ORDER)
_hdr_tot = "Dataset Name\t" + _headers + "\tTotal"

print("QA_DIR =", _qa_dir)
print("SPLITS_DIR (merged CSV) =", _spl)
print(f"merged_train 행 수: {len(_csv_tr_rows):,}  merged_test 행 수: {len(_csv_te_rows):,}")

print("\n" + "=" * 80)
print("(F1) 벤치마크 split = merged_train / merged_test CSV (논문 표 Split 행은 이것과 일치)")
print("-" * 80)
print(
    "※ (F2)와 숫자가 다른 이유: QA jsonl은 Task1/2/3·single/multi·instruction·CoT·stepwise 등"
    " 파일마다 같은 분자 쌍이 반복되어 행 수가 (CSV 행 수)×(과제 수)만큼 커짐.\n"
)

_csv_ep_union = [len(_csv_ep_tr[i] | _csv_ep_te[i]) for i in internals]
_csv_ep_only_tr = [len(_csv_ep_tr[i]) for i in internals]
_csv_ep_only_te = [len(_csv_ep_te[i]) for i in internals]
_csv_tot = [_csv_n_tr[i] + _csv_n_te[i] for i in internals]
_csv_tr = [_csv_n_tr[i] for i in internals]
_csv_te = [_csv_n_te[i] for i in internals]

print("[탭 복사 — 벤치마크 CSV 기준]")
print(_hdr_tot)
print(_row("Endpoint N", _csv_ep_union))
print()
print("Split")
print(_row("Total", _csv_tot))
print(_row("Train", _csv_tr))
print(_row("Test", _csv_te))
print("\n--- CSV 기준 split별 고유 endpoint ---")
print(_row("Endpoint N (Train)", _csv_ep_only_tr))
print(_row("Endpoint N (Test)", _csv_ep_only_te))

# ----- QA jsonl 합계 (참고) -----
_ep_all = [len(_endpoints_all.get(i, set())) for i in internals]
_ep_qa_tr = [len(_endpoints_train.get(i, set())) for i in internals]
_ep_qa_te = [len(_endpoints_test.get(i, set())) for i in internals]
_qa_tot = [_qa_train[i] + _qa_test[i] for i in internals]
_qa_tr = [_qa_train[i] for i in internals]
_qa_te = [_qa_test[i] for i in internals]

print("\n" + "=" * 80)
print("(F2) both_repre QA jsonl «행 수 합계» (모든 과제·스텝 파일 누적 — 벤치마크 행 수 아님)")
print("-" * 80)
print(_hdr_tot)
print(_row("Endpoint N (QA에서 본 고유 ep)", _ep_all))
print()
print("Split (QA 줄 합)")
print(_row("Total", _qa_tot))
print(_row("Train", _qa_tr))
print(_row("Test", _qa_te))
print("\n--- QA jsonl 기준 split별 고유 endpoint ---")
print(_row("Endpoint N (Train)", _ep_qa_tr))
print(_row("Endpoint N (Test)", _ep_qa_te))

known = {internal for _, internal in DATASET_DISPLAY_ORDER}
extra = sorted(set(_endpoints_all.keys()) - known)
if extra:
    print("\n(참고) 표에 없는 dataset_name (QA 쪽):")
    for k in extra:
        print(
            f"  {k}: endpoints(all)={len(_endpoints_all[k])}  QA train={_qa_train[k]} test={_qa_test[k]}"
        )

QA_DIR = /Users/jang-wonjun/Desktop/DMISLab/ToxAgent/ace_safe_ver/QA
SPLITS_DIR (merged CSV) = /Users/jang-wonjun/Desktop/DMISLab/ToxAgent/ace_safe_ver/splits/scaffold_by_endpoint_unseen_ver
merged_train 행 수: 57,008  merged_test 행 수: 6,243

(F1) 벤치마크 split = merged_train / merged_test CSV (논문 표 Split 행은 이것과 일치)
--------------------------------------------------------------------------------
※ (F2)와 숫자가 다른 이유: QA jsonl은 Task1/2/3·single/multi·instruction·CoT·stepwise 등 파일마다 같은 분자 쌍이 반복되어 행 수가 (CSV 행 수)×(과제 수)만큼 커짐.

[탭 복사 — 벤치마크 CSV 기준]
Dataset Name	hERG	Tox21	Sider	AMES	Clintox	DICTrank	DIRIL	DILIst	Skin Reaction	Metabolism	Total
Endpoint N	1	12	27	1	1	1	1	1	1	5	51

Split
Total	5520	37342	2528	7490	54	40	7	101	137	10032	63251
Train	4968	34527	1525	6780	0	0	0	71	107	9030	57008
Test	552	2815	1003	710	54	40	7	30	30	1002	6243

--- CSV 기준 split별 고유 endpoint ---
Endpoint N (Train)	1	12	17	1	0	0	0	1	1	5	38
Endpoint N (Test)	1	12	27	1	1	1	1	1	1	5	51

(F2) both_repre QA jsonl «행 수 합계» (모든 과제·스텝

In [16]:
# --- Task별 QA 개수 (both_repre만, agentic 제외) ---
# Task3 instruction / CoT: 동일 폴더 내 파일명으로 구분 (instruction vs CoT jsonl)

from pathlib import Path
import json
from collections import defaultdict

DATASET_COLS = [
    ("hERG", "herg_unified"),
    ("Tox21", "tox21_df"),
    ("Sider", "sider"),
    ("AMES", "ames"),
    ("Clintox", "clintox"),
    ("DICTrank", "dictrank"),
    ("DIRIL", "diril"),
    ("DILIst", "dilist"),
    ("Skin Reaction", "skin_reaction"),
    ("Metabolism", "metabolism"),
]
INTERNALS = [x[1] for x in DATASET_COLS]
DISPLAYS = [x[0] for x in DATASET_COLS]


def _qa_root() -> Path:
    cwd = Path.cwd()
    for c in (cwd, cwd / "ace_safe_ver" / "QA", cwd.parent / "QA" if cwd.name == "ace_safe_ver" else None):
        if c and c.is_dir() and (c / "test").is_dir() and (c / "train").is_dir():
            return c.resolve()
    for anc in [cwd, *cwd.parents]:
        q = anc / "ace_safe_ver" / "QA"
        if q.is_dir() and (q / "test").is_dir() and (q / "train").is_dir():
            return q.resolve()
    return cwd.resolve()


def _jl(p: Path):
    with open(p, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue


def _task_paths(split: str, task_dir: str, step: str) -> list[Path]:
    """both_repre만, agentic_flow_qa 하위 제외."""
    base = _qa_root() / split / task_dir / "both_repre" / step
    if not base.is_dir():
        return []
    return sorted(base.glob("*_qa.jsonl"))


def _count_by_dataset(files: list[Path]) -> dict[str, int]:
    out = defaultdict(int)
    for fp in files:
        for o in _jl(fp):
            ds = o.get("dataset_name")
            if not ds:
                continue
            out[str(ds)] += 1
    return out


def _count_by_glob(split: str, task_dir: str, step: str, pattern: str) -> dict[str, int]:
    """task_dir/both_repre/step 아래 glob 패턴에 맞는 jsonl만 집계 (파일명으로 instruction vs CoT 구분)."""
    base = _qa_root() / split / task_dir / "both_repre" / step
    if not base.is_dir():
        return defaultdict(int)
    files = [p for p in base.glob(pattern) if p.is_file()]
    if not files:
        return defaultdict(int)
    return _count_by_dataset(files)


T3_INSTRUCTION_DIR = "task3_instruction_nontoxic_smiles_generation"


def _row_vals(cnt: dict[str, int]) -> list[int]:
    return [cnt.get(i, 0) for i in INTERNALS]


def _print_block(title: str, rows: list[tuple[str, dict[str, int]]]):
    hdr = "\t".join(DISPLAYS + ["Total"])
    print("\n" + "=" * 80)
    print(title)
    print("-" * 80)
    print(hdr)
    for label, cnt in rows:
        vals = _row_vals(cnt)
        tot = sum(vals)
        print(label + "\t" + "\t".join(str(v) for v in vals) + "\t" + str(tot))


ROOT = _qa_root()
print("QA_DIR =", ROOT)
print("조건: both_repre | agentic 제외")
print("Task3 instruction/CoT: task3_instruction_nontoxic_smiles_generation/both_repre/{single,multi}_step/")
print("  - 파일명 *instruction*nontoxic*qa.jsonl → instruction")
print("  - 파일명 *CoT*nontoxic*qa.jsonl → CoT\n")

for split in ("train", "test"):
    t1s = _count_by_dataset(_task_paths(split, "task1_toxic_fragment_identification", "single_step"))
    t1m = _count_by_dataset(_task_paths(split, "task1_toxic_fragment_identification", "multi_step"))
    t2s = _count_by_dataset(_task_paths(split, "task2_nontoxic_fragment_generation", "single_step"))
    t2m = _count_by_dataset(_task_paths(split, "task2_nontoxic_fragment_generation", "multi_step"))
    t3s = _count_by_dataset(_task_paths(split, "task3_nontoxic_smiles_generation", "single_step"))
    t3m = _count_by_dataset(_task_paths(split, "task3_nontoxic_smiles_generation", "multi_step"))
    # task3_instruction_*_qa.jsonl / task3_CoT_*_qa.jsonl (train·test 각각 둘 다 있으면 각각 집계)
    t3instr_s = _count_by_glob(split, T3_INSTRUCTION_DIR, "single_step", "*instruction*nontoxic*qa.jsonl")
    t3instr_m = _count_by_glob(split, T3_INSTRUCTION_DIR, "multi_step", "*instruction*nontoxic*qa.jsonl")
    t3cot_s = _count_by_glob(split, T3_INSTRUCTION_DIR, "single_step", "*CoT*nontoxic*qa.jsonl")
    t3cot_m = _count_by_glob(split, T3_INSTRUCTION_DIR, "multi_step", "*CoT*nontoxic*qa.jsonl")

    _print_block(
        f"QA for {split.upper()} (both_repre, agentic 제외)",
        [
            ("Task1 Single Step", t1s),
            ("Task1 Multi Step", t1m),
            ("Task2 Single Step", t2s),
            ("Task2 Multi Step", t2m),
            ("Task3 (vanilla) Single Step", t3s),
            ("Task3 (vanilla) Multi Step", t3m),
            ("Task3 instruction Single Step", t3instr_s),
            ("Task3 instruction Multi Step", t3instr_m),
            ("Task3 CoT Single Step", t3cot_s),
            ("Task3 CoT Multi Step", t3cot_m),
        ],
    )

print(
    "\n※ Subtask1/2는 both_repre 하위에 없어 위 표에 넣지 않음. "
    "개수는 subtask*_qa.jsonl 전체 줄 수로 별도 확인."
)

QA_DIR = /Users/jang-wonjun/Desktop/DMISLab/ToxAgent/ace_safe_ver/QA
조건: both_repre | agentic 제외
Task3 instruction/CoT: task3_instruction_nontoxic_smiles_generation/both_repre/{single,multi}_step/
  - 파일명 *instruction*nontoxic*qa.jsonl → instruction
  - 파일명 *CoT*nontoxic*qa.jsonl → CoT


QA for TRAIN (both_repre, agentic 제외)
--------------------------------------------------------------------------------
hERG	Tox21	Sider	AMES	Clintox	DICTrank	DIRIL	DILIst	Skin Reaction	Metabolism	Total
Task1 Single Step	1854	6147	372	2432	0	0	0	16	16	1532	12369
Task1 Multi Step	3114	28380	1153	4348	0	0	0	55	91	7498	44639
Task2 Single Step	1759	4342	367	1701	0	0	0	13	10	1365	9557
Task2 Multi Step	3209	30185	1158	5079	0	0	0	58	97	7665	47451
Task3 (vanilla) Single Step	1759	4342	367	1701	0	0	0	13	10	1365	9557
Task3 (vanilla) Multi Step	3209	30185	1158	5079	0	0	0	58	97	7665	47451
Task3 instruction Single Step	1759	4342	367	1701	0	0	0	13	10	1365	9557
Task3 instruction Multi Step	3209	30185	1158	5079	0	0	0	5

In [17]:
# --- multi_step 분류 근거(= dot-sep fragment 개수) 분포 ---
# 기준: build_safe_qa.py의 _classify_step() — only_*_safe_fragments에서 dot('.')로 2개 이상이면 multi_step

from pathlib import Path
import json
import re
from collections import Counter, defaultdict


def _qa_dir() -> Path:
    cwd = Path.cwd()
    for c in (cwd, cwd / "ace_safe_ver" / "QA", cwd.parent / "QA" if cwd.name == "ace_safe_ver" else None):
        if c and c.is_dir() and (c / "test").is_dir() and (c / "train").is_dir():
            return c.resolve()
    for anc in [cwd, *cwd.parents]:
        q = anc / "ace_safe_ver" / "QA"
        if q.is_dir() and (q / "test").is_dir() and (q / "train").is_dir():
            return q.resolve()
    return cwd.resolve()


def _iter_jsonl(p: Path):
    with open(p, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue


def _count_dot_frags(s: str | None) -> int:
    if not s:
        return 0
    parts = [x.strip() for x in str(s).split(".") if x.strip()]
    return len(parts)


# task2 질문에서 only_toxic_safe_fragments 문구 추출 (multi_step일 때는 candidates are: frag1.frag2 형태)
_RE_T2_TOX = re.compile(r"candidates? for toxicity-associated structure[^\n]*?are:\s*([^\n]+)", re.IGNORECASE)


def _infer_task_from_path(fp: Path) -> str:
    rel = fp.relative_to(QA_DIR).parts
    # rel: split/task_dir/both_repre/multi_step/file
    for seg in rel:
        if seg.startswith("task") or seg.startswith("subtask"):
            return seg
    return "(unknown)"


def extract_evidence_frag_count(fp: Path, obj: dict) -> int | None:
    """multi_step 분류 근거가 되는 fragment 개수(가능한 경우 max count)."""
    task_dir = _infer_task_from_path(fp)

    # Task1: answer가 only_toxic_safe_fragments
    if task_dir.startswith("task1_"):
        ans = (obj.get("answer") or {}).get("answer")
        return _count_dot_frags(ans)

    # Task2: question에 only_toxic_safe_fragments가 있고, answer가 only_nontoxic_safe_fragments
    if task_dir.startswith("task2_"):
        q = obj.get("question") or ""
        m = _RE_T2_TOX.search(q)
        tox = m.group(1).strip() if m else ""
        non = (obj.get("answer") or {}).get("answer")
        return max(_count_dot_frags(tox), _count_dot_frags(non))

    # Task3 stepwise CoT: answer에 gold_only_*_safe_fragments 존재
    if task_dir.startswith("task3_stepwise_cot_"):
        a = obj.get("answer") or {}
        tox = a.get("gold_only_toxic_safe_fragments")
        non = a.get("gold_only_nontoxic_safe_fragments")
        return max(_count_dot_frags(tox), _count_dot_frags(non))

    # 그 외(Task3 vanilla / instruction 등)는 QA에 근거 fragment 문자열이 없어 계산 불가
    return None


QA_DIR = _qa_dir()
print("QA_DIR =", QA_DIR)

splits = ("train", "test")

# both_repre/multi_step만 대상으로 분포 계산 (agentic_flow_qa 제외)
paths: list[Path] = []
for sp in splits:
    root = QA_DIR / sp
    if not root.is_dir():
        continue
    for fp in root.rglob("*_qa.jsonl"):
        rel = fp.relative_to(QA_DIR).parts
        if "agentic_flow_qa" in rel:
            continue
        if "both_repre" not in rel:
            continue
        if "multi_step" not in rel:
            continue
        paths.append(fp)

by_split_task = defaultdict(Counter)  # (split, task_dir) -> Counter[count]
by_split_all = defaultdict(Counter)   # split -> Counter[count]
missing = Counter()  # task_dir -> n

for fp in paths:
    sp = fp.relative_to(QA_DIR).parts[0]
    task_dir = _infer_task_from_path(fp)
    for o in _iter_jsonl(fp):
        n = extract_evidence_frag_count(fp, o)
        if n is None:
            missing[task_dir] += 1
            continue
        by_split_task[(sp, task_dir)][n] += 1
        by_split_all[sp][n] += 1


def _print_counter(title: str, c: Counter):
    if not c:
        print(title + ": (empty)")
        return
    total = sum(c.values())
    keys = sorted(c.keys())
    print(title + f"  (n={total:,})")
    for k in keys:
        print(f"  fragments={k}: {c[k]:,}  ({c[k]/total:.2%})")


print("\n" + "=" * 80)
print("multi_step 근거 fragment 개수 분포 (both_repre/multi_step만)")
print("-" * 80)
for sp in splits:
    _print_counter(f"[{sp}] 전체", by_split_all[sp])

print("\n" + "=" * 80)
print("Task별 분포")
print("-" * 80)
for (sp, task_dir), c in sorted(by_split_task.items(), key=lambda x: (x[0][0], x[0][1])):
    _print_counter(f"[{sp}] {task_dir}", c)

if missing:
    print("\n" + "=" * 80)
    print("(참고) QA에서 근거 fragment 문자열을 못 찾아 제외된 레코드 수")
    print("-" * 80)
    for k, v in missing.most_common():
        print(f"{k}: {v:,}")
    print("(task3_nontoxic_smiles_generation / task3_instruction_* 등은 QA에 근거 fragment가 없어 제외되는 것이 정상)")

QA_DIR = /Users/jang-wonjun/Desktop/DMISLab/ToxAgent/ace_safe_ver/QA

multi_step 근거 fragment 개수 분포 (both_repre/multi_step만)
--------------------------------------------------------------------------------
[train] 전체  (n=139,541)
  fragments=2: 41,354  (29.64%)
  fragments=3: 62,128  (44.52%)
  fragments=4: 36,059  (25.84%)
[test] 전체  (n=12,100)
  fragments=2: 5,616  (46.41%)
  fragments=3: 3,957  (32.70%)
  fragments=4: 2,526  (20.88%)
  fragments=6: 1  (0.01%)

Task별 분포
--------------------------------------------------------------------------------
[test] task1_toxic_fragment_identification  (n=3,872)
  fragments=2: 2,117  (54.67%)
  fragments=3: 1,199  (30.97%)
  fragments=4: 556  (14.36%)
[test] task2_nontoxic_fragment_generation  (n=4,114)
  fragments=2: 1,749  (42.51%)
  fragments=3: 1,379  (33.52%)
  fragments=4: 985  (23.94%)
  fragments=6: 1  (0.02%)
[test] task3_stepwise_cot_nontoxic_smiles_generation  (n=4,114)
  fragments=2: 1,750  (42.54%)
  fragments=3: 1,379  (33.52%)
  